# Pass 1: Grief & Outrage + Context
LLM coding of grief/outrage in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Sentiment labels (POSITIVE / NEGATIVE / NEUTRAL) quantify tone but collapse
moral emotions. Pass 1 isolates Grief and Outrage specifically, plus the
value/relationship invoked as grounds and whose family is referenced —
the moral core of crisis discourse.

### Approach
Chunked LLM calls over slim `df[["video_id", "text"]]` CSV pasted into the
versioned prompt in `../prompts/pass-01-grief-outrage.md`. Output strictly
JSON: `grief_or_outrage ⊆ {Grief, Outrage}`, `value_or_relationship_invoked`,
`family_referenced[]`. Empty array / "None" when absent.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Load** | Read micro-sample | Preprocessed CSV → `n_sample=30` rows |
| **2. Preprocess** | Slim + clean | Keep `video_id, text`; `dropna`; `strip` |
| **3. Chunk** | Split rows | `chunk_size=15` → `chunk.to_csv(index=False)` per call |
| **4. Call** | LLM coding | `generate_content`, strip fences, `json.loads`, retry-once |
| **5. Export** | Save + inspect | `sample_pass01_results.jsonl` → merge on `row_id` for QA |

### Cross-Analysis
- *Which videos concentrate outrage vs grief?*
- *What values/relationships ground each emotion?*
- *Whose family is invoked, and where?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/sample_pass01_results.jsonl`

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [14]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import random

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 200
chunk_size = 5
sample_seed = 42
model = "gemma-4-31b-it"
overwrite = False

In [19]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
OUTPUT_PATH = f"../output/{crisis_name}/sample_pass01_results.jsonl"
LOG_PATH    = f"../output/{crisis_name}/sample_pass01_failures.log"

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemma-4-31b-it" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 8192  # max output ceiling; model uses only what it needs
BATCH_SIZE  = chunk_size  # 15 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)

✅ Gemma API key loaded.


## Micro-sample ingest & preprocess
Load `n_sample` rows from `INPUT_PATH`, slim to `video_id` + `text`,
drop nulls, strip whitespace. Keeps `row_id` for lossless merge at inspect time.

In [20]:
full_df = pd.read_csv(INPUT_PATH)
print(f"Full phase rows: {len(full_df)}")

sample_df = full_df.head(n_sample).copy()  # deterministic for review; n_sample=200
sample_df = sample_df[["video_id", "text"]]
sample_df = sample_df.dropna()
sample_df["text"] = sample_df["text"].astype(str).str.strip()
sample_df = sample_df[sample_df["text"] != ""].reset_index(drop=True)
sample_df["row_id"] = sample_df.index
print(f"Sample rows: {len(sample_df)}")
sample_df.head()

Full phase rows: 1411
Sample rows: 200


,video_id,text,row_id
0,dtvrM51u4Ks,an eye for an eye and tooth for tooth.stop thi...,0
1,dtvrM51u4Ks,justice for the 2 young guys and no one will g...,1
2,dtvrM51u4Ks,if he apologizes that means guilty in the deat...,2
3,dtvrM51u4Ks,you should stay here in the philippines until ...,3
4,dtvrM51u4Ks,did you go to their wake?,4


## Batch chunking
Slice the sample into 15-row chunks and convert each chunk to a raw CSV string
for one LLM call apiece.

In [21]:
chunks = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
chunk_csvs = [c[["video_id", "text"]].to_csv(index=False) for c in chunks]
print(f"{len(chunks)} chunks x ~{BATCH_SIZE} rows | first chunk chars: {len(chunk_csvs[0])}")
print(chunk_csvs[0][:600])

14 chunks x ~15 rows | first chunk chars: 2104
video_id,text
dtvrM51u4Ks,"an eye for an eye and tooth for tooth.stop this scripted lies, go to hell"
dtvrM51u4Ks,justice for the 2 young guys and no one will go thru this
dtvrM51u4Ks,if he apologizes that means guilty in the death of 2 players
dtvrM51u4Ks,you should stay here in the philippines until justice given to rene and divine and to their berieved family. don't hide. face their families personally.
dtvrM51u4Ks,did you go to their wake?
dtvrM51u4Ks,si coach tab ang nag papatunay na basta kalbo masamang tao
dtvrM51u4Ks,"surrender yourself and admit your mistake, go to jail, that was the 


## LLM inference
Pass the system instruction + one CSV chunk per call to the Google AI Studio API.
Sequential calls with a 1s pause; raw responses kept in memory for the parse step.

In [22]:
prompt_template = Path("../prompts/pass-01-grief-outrage.txt").read_text(encoding="utf-8").strip()
print(prompt_template[:400])

def call_with_retry(full_input, tries=4):
    for attempt in range(tries):
        try:
            return client.models.generate_content(model=MODEL, contents=full_input,
                config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS))
        except Exception as e:
            if attempt == tries - 1:
                raise
            sleep_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"  [Attempt {attempt+1} failed: {repr(e)}] Retrying in {sleep_time:.2f}s...")
            time.sleep(sleep_time)

chunks_queue = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
raw_responses, hard_failures = [], []
print(f"Starting dynamic queue with {len(chunks_queue)} initial chunks...")

while chunks_queue:
    current_chunk = chunks_queue.pop(0)
    n_rows = len(current_chunk)
    csv_str = current_chunk[["video_id", "text"]].to_csv(index=False)
    try:
        resp = call_with_retry(prompt_template + "\n\nData:\n" + csv_str)
        raw_responses.append({"row_ids": current_chunk["row_id"].tolist(), "text": resp.text or ""})
        time.sleep(1)
    except Exception as e:
        print(f"❌ Chunk failure (Size: {n_rows}): {repr(e)}")
        if n_rows > 1:
            print(f"   ↳ Splitting chunk of {n_rows}...")
            mid = n_rows // 2
            chunks_queue.insert(0, current_chunk.iloc[mid:])
            chunks_queue.insert(0, current_chunk.iloc[:mid])
        else:
            row = current_chunk.iloc[0]
            print(f"   ↳ Hard failure on row_id {row['row_id']}. Logging and skipping.")
            hard_failures.append({"row_id": int(row["row_id"]), "video_id": row.get("video_id"),
                                  "error": repr(e), "text_length": len(str(row["text"]))})

print(f"\nDone. {len(raw_responses)} ok chunks, {len(hard_failures)} hard failures.")

System Instruction:
I will provide a dataset of comments in CSV format containing `video_id` and `text` columns. You must return the analysis strictly as a JSON array of objects. Do not include markdown formatting (other than the JSON block) or explanations outside of the JSON.

For each comment expressing grief or outrage in the text column, extract:
(a) the emotion (specifically whether it is Gr


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemma-3-4b-it is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

## API probe (run once)
Single-row call to isolate model/key issues from chunk payload issues before the full loop.

In [ ]:
probe = prompt_template + "\n\n" + sample_df.head(1)[["video_id", "text"]].to_csv(index=False)
r = client.models.generate_content(
    model=MODEL,
    contents=probe,
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=2048),
)
print((r.text or "")[:1500])